# DASTRP-Net — inference demo

This notebook runs the pre-trained DASTRP-Net on a dashcam video from the
IMBD-ESR dataset and shows

1. the per-frame **risk level** (Safe / Warning / Danger) with class probabilities,
2. the **evidential uncertainty** $u = K / S$ of each prediction,
3. detected motorcycles with their predicted motion state and per-object risk,
4. an annotated output video.

**Before running**

* install the environment (`docs/INSTALL.md`),
* get the weights (`python tools/download_weights.py`, done automatically below if missing),
* get one IMBD-ESR video, e.g. the sample `videos/ugm/ugm_1_1.mp4` from
  <https://github.com/SaveVic/dastrp-dataset>, into `data/imbd-esr/`.

Runs on CPU; a GPU is used automatically when available.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

# repository root = parent of notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from utils.reproducibility import set_seed
from scripts.inference import DASTRPInferencer, run_video
from data.dataset import infer_domain, DOMAIN_NAMES, RISK_NAMES

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", ROOT, "| device:", DEVICE, "| torch", torch.__version__)

## 2. Configuration

Edit the paths if your files live elsewhere (environment variables with the same
names override the defaults).

In [ ]:
CHECKPOINT = os.environ.get("CHECKPOINT", "weights/dastrp_exp3.pth")
YOLO       = os.environ.get("YOLO",       "weights/yolo11n_motor.pt")
VIDEO      = os.environ.get("VIDEO",      "data/imbd-esr/videos/ugm/ugm_1_1.mp4")
IMG_SIZE   = int(os.environ.get("IMG_SIZE", 640))   # input size used in the paper
T          = 4                                       # clip length used in the paper
MAX_FRAMES = int(os.environ.get("MAX_FRAMES", 300))  # frames processed in section 5
OUT_DIR    = Path(os.environ.get("OUT_DIR", "runs/demo"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [p for p in (CHECKPOINT, YOLO) if not Path(p).exists()]
if missing:
    print("Downloading weights:", missing)
    subprocess.run([sys.executable, "tools/download_weights.py", "--only",
                    *[Path(p).name for p in missing]], check=True)

assert Path(VIDEO).exists(), f"Video not found: {VIDEO} (see the notes at the top)"
domain_id = infer_domain(VIDEO)
print("video :", VIDEO, "| domain from file name:", DOMAIN_NAMES.get(domain_id, "unknown"))

## 3. Load the model

In [ ]:
inf = DASTRPInferencer(
    checkpoint   = CHECKPOINT,
    yolo_weights = YOLO,
    T            = T,
    img_size     = IMG_SIZE,
    device       = DEVICE,
)
n_params = sum(p.numel() for p in inf.model.parameters())
print(f"parameters: {n_params / 1e6:.2f} M")

## 4. One clip

The inferencer keeps a sliding buffer of the last `T` frames; every new frame
produces a prediction for the clip that ends at that frame.

In [ ]:
cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 7.9
frames = []
for _ in range(T):
    ok, fr = cap.read()
    if not ok:
        break
    frames.append(fr)
cap.release()

inf.reset()
for fr in frames:
    res = inf.predict_frame(fr, domain_id=domain_id if domain_id >= 0 else None)

print("risk        :", RISK_NAMES[res["risk_id"]])
print("probability :", {k: round(float(v), 3) for k, v in zip(RISK_NAMES.values(), res["risk_prob"])})
print("Dirichlet α :", res["alpha"].round(3))
print("uncertainty :", round(res["uncertainty"], 3), "(0 = confident, 1 = no evidence)")
print("motorcycles :", len(res["boxes"]))

fig, axes = plt.subplots(1, T, figsize=(4 * T, 3))
for t, (ax, fr) in enumerate(zip(axes, frames)):
    img = inf.annotate(fr.copy(), res) if t == T - 1 else fr
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_axis_off()
    ax.set_title(f"frame {t}" + ("  (prediction)" if t == T - 1 else ""))
plt.tight_layout(); plt.show()

## 5. Risk and uncertainty over time

In [ ]:
rows, keep = [], {}
cap = cv2.VideoCapture(VIDEO)
inf.reset()
for i in range(MAX_FRAMES):
    ok, fr = cap.read()
    if not ok:
        break
    r = inf.predict_frame(fr, domain_id=domain_id if domain_id >= 0 else None)
    rows.append({"frame": i, "time_s": i / fps, "risk": r["risk_id"],
                 "p_safe": r["risk_prob"][0], "p_warning": r["risk_prob"][1],
                 "p_danger": r["risk_prob"][2], "uncertainty": r["uncertainty"],
                 "n_motor": len(r["boxes"])})
    # keep the most risky frame (highest danger + warning probability)
    score = r["risk_prob"][1] + r["risk_prob"][2]
    if score >= keep.get("score", -1):
        keep = {"score": score, "frame": fr.copy(), "res": r, "idx": i}
cap.release()

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "timeline.csv", index=False)
print(df["risk"].map(RISK_NAMES).value_counts().to_string())
df.head()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.stackplot(df.time_s, df.p_safe, df.p_warning, df.p_danger,
              labels=["Safe", "Warning", "Danger"],
              colors=["#4caf50", "#ffb300", "#e53935"], alpha=0.85)
ax1.set_ylabel("class probability"); ax1.set_ylim(0, 1)
ax1.legend(loc="upper right", ncol=3, fontsize=8)
ax2.plot(df.time_s, df.uncertainty, color="#5e35b1")
ax2.set_ylabel("uncertainty u"); ax2.set_xlabel("time (s)"); ax2.set_ylim(0, 1)
ax2b = ax2.twinx()
ax2b.bar(df.time_s, df.n_motor, width=1 / fps, color="grey", alpha=0.25)
ax2b.set_ylabel("# motorcycles", color="grey"); ax2b.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

## 6. Most risky frame

In [ ]:
r = keep["res"]
img = inf.annotate(keep["frame"].copy(), r)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"frame {keep['idx']}: {RISK_NAMES[r['risk_id']]}  "
          f"p={r['risk_prob'].round(2)}  u={r['uncertainty']:.2f}")
plt.show()

if len(r["boxes"]):
    print(pd.DataFrame({
        "cx": r["boxes"][:, 0], "cy": r["boxes"][:, 1],
        "w": r["boxes"][:, 2], "h": r["boxes"][:, 3],
        "motion": ["moving" if m == 0 else "stationary" for m in r["motion_pred"]],
        "object risk": [RISK_NAMES[int(k)] for k in r["obj_risk_pred"]],
    }).round(3).to_string())

## 7. Annotated video

Writes the full video with the risk overlay and a per-frame CSV log
(same as `python scripts/inference.py ...`).

In [ ]:
out_video = OUT_DIR / (Path(VIDEO).stem + "_annotated.mp4")
inf.reset()
run_video(inf, VIDEO, dst=str(out_video), log_csv=str(OUT_DIR / "frames_log.csv"),
          domain_id=domain_id if domain_id >= 0 else None)
print("saved:", out_video)

## Notes

* `risk_prob` is the mean of the Dirichlet distribution, $\alpha_k / S$ with $S = \sum_k \alpha_k$;
  the uncertainty is $u = K / S$ ($K = 3$). High $u$ means little evidence for any class.
* The road domain (UGM / Utara / Ringroad) is taken from the file name; pass
  `domain_id=None` to let the scene classifier decide.
* Reported metrics in the paper are computed with `scripts/evaluate.py` on the
  test split, not with this per-frame streaming demo.